# DAPD Pipeline — Google Colab 실행 노트북

**Domain-Adaptive Progressive Distillation (DAPD)**

---

## 사전 확인 사항

| 항목 | 권장 사양 | 비고 |
|---|---|---|
| **GPU** | A100 40GB (Colab Pro) | T4로도 v2 config 실행 가능 |
| **RAM** | 52GB+ (High RAM) | T4는 12.7GB |
| **Disk** | 100GB+ | 모델 + 체크포인트 저장 |
| **런타임** | 24h (Pro) / 12h (Free) | full-scale은 Pro 필수 |

> **런타임 → 런타임 유형 변경**에서 GPU를 A100으로 설정 후 시작하세요.

---
## 0. GPU 및 환경 확인

In [ ]:
import subprocess, sys

# GPU 정보 출력
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                        '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip())

# 디스크 용량 확인
result = subprocess.run(['df', '-h', '/'], capture_output=True, text=True)
print('Disk:')
print(result.stdout)

---
## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 프로젝트가 저장된 Drive 경로 설정
# ★ 본인 환경에 맞게 수정하세요
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/Domain_Adaption'

import os
if not os.path.exists(DRIVE_PROJECT_PATH):
    raise FileNotFoundError(
        f'Drive 경로를 찾을 수 없습니다: {DRIVE_PROJECT_PATH}\n'
        'DRIVE_PROJECT_PATH 변수를 실제 경로로 수정하세요.'
    )

print(f'✅ Drive 마운트 완료: {DRIVE_PROJECT_PATH}')
print('프로젝트 내 파일 목록:')
for f in sorted(os.listdir(DRIVE_PROJECT_PATH)):
    print(f'  {f}')

---
## 2. 프로젝트 디렉토리 설정 및 작업 디렉토리 구성

Colab 로컬(`/content/dapd`)에 작업 복사본을 만들고,
결과(runs/)만 Google Drive에 실시간 동기화합니다.

- **이유**: Drive I/O는 느리므로 학습 중간 체크포인트는 로컬에 쓰고, 최종 결과만 Drive에 저장

In [ ]:
import shutil, os

# 작업 디렉토리 (Colab 로컬 — 빠른 I/O)
WORK_DIR = '/content/dapd'

# Drive 결과 저장 경로
DRIVE_RUNS_PATH = os.path.join(DRIVE_PROJECT_PATH, 'runs')
DRIVE_CACHE_PATH = os.path.join(DRIVE_PROJECT_PATH, 'cache')

# 소스코드 복사 (runs/, cache/ 제외)
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

def copytree_selective(src, dst, exclude=('runs', 'cache', '__pycache__', '.git')):
    os.makedirs(dst, exist_ok=True)
    for item in os.listdir(src):
        if item in exclude:
            continue
        s = os.path.join(src, item)
        d = os.path.join(dst, item)
        if os.path.isdir(s):
            copytree_selective(s, d, exclude)
        else:
            shutil.copy2(s, d)

copytree_selective(DRIVE_PROJECT_PATH, WORK_DIR)

# runs/ → Drive로 심볼릭 링크 (결과 자동 Drive 저장)
os.makedirs(DRIVE_RUNS_PATH, exist_ok=True)
os.makedirs(DRIVE_CACHE_PATH, exist_ok=True)

runs_local = os.path.join(WORK_DIR, 'runs')
cache_local = os.path.join(WORK_DIR, 'cache')

if not os.path.exists(runs_local):
    os.symlink(DRIVE_RUNS_PATH, runs_local)
if not os.path.exists(cache_local):
    os.symlink(DRIVE_CACHE_PATH, cache_local)

os.chdir(WORK_DIR)
print(f'✅ 작업 디렉토리: {WORK_DIR}')
print(f'✅ runs/ → {DRIVE_RUNS_PATH} (Drive 실시간 저장)')
print(f'✅ cache/ → {DRIVE_CACHE_PATH} (HuggingFace 캐시)')

---
## 3. 패키지 설치

In [ ]:
# DAPD 패키지 및 의존성 설치
!pip install -q -e /content/dapd

# 추가 설치 (Colab 기본 설치와 버전 충돌 방지)
!pip install -q \
    'transformers>=4.42.0' \
    'peft>=0.11.0' \
    'accelerate>=0.30.0' \
    'datasets>=2.19.0' \
    'evaluate>=0.4.2' \
    'fvcore>=0.1.5.post20221221' \
    'scikit-learn>=1.4.2'

print('✅ 패키지 설치 완료')

In [ ]:
# 환경 검증 (FIX-01 ~ FIX-04 포함)
import sys
sys.path.insert(0, '/content/dapd/src')
os.chdir('/content/dapd')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

# FIX 검증
from dapd.metrics.core import _answer_matches
print('FIX-01 _answer_matches: OK')
from dapd.pruning import _prune_attention_heads
print('FIX-02 GQA pruning: OK')
from dapd.data import _map_bioasq
r = _map_bioasq({'question': 'Test?', 'text': '<answer> BRCA1 <context> context text'})
assert r['target'] == 'BRCA1'
print('FIX-03 BioASQ mapper: OK')
from dapd.analysis import create_dynamics_callback
print('FIX-04 dynamics callback: OK')
print('\n✅ 모든 환경 검증 통과')

---
## 4. Colab 전용 Config 생성

`dapd_full.yaml`의 경로를 Colab 환경에 맞게 재설정합니다.

- `cache_dir` → Drive 캐시 경로
- `output_dir` → Colab 로컬 runs/ (Drive 심볼릭 링크)
- `device` → cuda (GPU)
- `num_proc` → Colab CPU 코어 수에 맞게 조정

In [ ]:
import yaml, copy, os

# ── 실험 종류 선택 ────────────────────────────────────────────────────────
# 'v2'   : 5k 샘플, 128 토큰 (T4 무료로도 실행 가능, 15분/회)
# 'full' : 전체 데이터, 256 토큰 (A100 권장, 49분/회)
EXPERIMENT_SCALE = 'full'   # ★ 'v2' 또는 'full'

# ── 기본 config 로드 ──────────────────────────────────────────────────────
base_config_path = {
    'v2':   '/content/dapd/configs/dapd_m4_16gb.yaml',
    'full': '/content/dapd/configs/dapd_full.yaml',
}[EXPERIMENT_SCALE]

with open(base_config_path) as f:
    cfg = yaml.safe_load(f)

# ── Colab 환경에 맞게 경로/설정 수정 ─────────────────────────────────────
COLAB_RUNS = f'./runs/dapd_{EXPERIMENT_SCALE}'
HF_CACHE   = f'./cache/hf'

# data
cfg['data']['cache_dir'] = HF_CACHE
cfg['data']['tokenized_cache_dir'] = f'{COLAB_RUNS}/cache/tokenized'
cfg['data']['num_proc'] = 2  # Colab CPU 코어 안전값

# adaptation output
cfg['adaptation']['output_dir'] = f'{COLAB_RUNS}/domain_teacher'

# distillation output
cfg['distillation']['output_dir'] = f'{COLAB_RUNS}/distilled_student'

# pruning output
cfg['pruning']['output_dir'] = f'{COLAB_RUNS}/pruned_student'

# evaluation output
cfg['evaluation']['output_file'] = f'{COLAB_RUNS}/eval_metrics.json'
cfg['evaluation']['ood_output_file'] = f'{COLAB_RUNS}/eval_metrics_ood.json'

# analysis output
cfg['analysis']['output_dir'] = f'{COLAB_RUNS}/analysis'

# runtime: CUDA 사용
cfg['runtime']['device'] = 'cuda'
cfg['runtime']['deterministic'] = True

# full scale 전용: bf16 활성화, analysis 전체 on
if EXPERIMENT_SCALE == 'full':
    cfg['adaptation']['bf16'] = True
    cfg['analysis']['run_teacher_distribution'] = True
    cfg['analysis']['run_temperature_analysis'] = True

# v2: M4 전용 설정 해제
if EXPERIMENT_SCALE == 'v2':
    cfg['adaptation']['optim'] = 'adamw_torch'  # Adafactor → AdamW (CUDA)
    cfg['adaptation']['bf16'] = True
    cfg['distillation']['optim'] = 'adamw_torch' if 'optim' in cfg.get('distillation', {}) else None
    # analysis — CUDA에서는 모두 활성화
    cfg['analysis']['run_teacher_distribution'] = True
    cfg['analysis']['run_temperature_analysis'] = True
    if cfg['analysis'].get('run_distillation_intervention') is None:
        cfg['analysis']['run_distillation_intervention'] = False

# Colab config 저장
colab_config_path = f'/content/dapd/configs/dapd_colab_{EXPERIMENT_SCALE}.yaml'
with open(colab_config_path, 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True, sort_keys=False)

print(f'✅ Colab config 생성: {colab_config_path}')
print(f'   scale: {EXPERIMENT_SCALE}')
print(f'   device: {cfg["runtime"]["device"]}')
print(f'   max_train_samples: {cfg["data"]["max_train_samples"]}')
print(f'   max_length: {cfg["data"]["max_length"]}')
print(f'   adapt_epochs: {cfg["adaptation"]["num_train_epochs"]}')
print(f'   distill_epochs: {cfg["distillation"]["num_train_epochs"]}')
print(f'   bf16: {cfg["adaptation"].get("bf16")}')

---
## 5. HuggingFace 모델 사전 다운로드 (선택)

학습 시작 전에 모델을 미리 다운로드해두면, 학습 중 네트워크 이슈를 방지할 수 있습니다.

In [ ]:
# 선택사항: 모델 사전 다운로드
# HuggingFace Hub에서 Teacher / Student 모델을 미리 캐시합니다

from transformers import AutoModelForCausalLM, AutoTokenizer
import os

os.environ['HF_HOME'] = '/content/dapd/cache/hf'

MODELS = [
    'Qwen/Qwen2.5-1.5B-Instruct',  # Teacher
    'Qwen/Qwen2.5-0.5B-Instruct',  # Student
]

for model_name in MODELS:
    print(f'Downloading {model_name}...')
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    # 모델 가중치만 다운로드 (실제 로드는 학습 시 수행)
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=model_name, cache_dir='/content/dapd/cache/hf')
    print(f'  ✅ {model_name} 다운로드 완료')

print('\n✅ 모든 모델 다운로드 완료')

---
## 6. 메인 파이프라인 실행

**Domain Adaptation → Teacher Logits → Progressive KD → Pruning → Evaluation**

| 설정 | v2 (T4) | full (A100) |
|---|---|---|
| 학습 샘플 | 5,000 | 전체 (~390k) |
| max_length | 128 | 256 |
| adapt + distill | 2 + 3 epoch | 3 + 5 epoch |
| 예상 시간 | ~15분 | ~49분 |

In [ ]:
import os, subprocess, sys

os.chdir('/content/dapd')
os.environ['PYTHONPATH'] = '/content/dapd/src'
os.environ['HF_HOME'] = '/content/dapd/cache/hf'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

EXPERIMENT_SCALE = 'full'  # ★ 위 섹션과 동일하게 설정
config_path = f'/content/dapd/configs/dapd_colab_{EXPERIMENT_SCALE}.yaml'

print(f'🚀 메인 파이프라인 시작: {EXPERIMENT_SCALE} scale')
print(f'   config: {config_path}')
print('   결과는 Google Drive에 실시간 저장됩니다.')
print('   진행 상황은 아래 로그를 확인하세요.')
print('-' * 60)

!PYTHONPATH=/content/dapd/src \
  HF_HOME=/content/dapd/cache/hf \
  TOKENIZERS_PARALLELISM=false \
  python /content/dapd/scripts/run_pipeline.py \
    --config {config_path} \
  2>&1 | tee /content/drive/MyDrive/Domain_Adaption/runs/pipeline_log.txt

In [ ]:
# 파이프라인 결과 확인
import json

EXPERIMENT_SCALE = 'full'  # ★
runs_dir = f'/content/dapd/runs/dapd_{EXPERIMENT_SCALE}'

eval_path = f'{runs_dir}/eval_metrics.json'
ood_path  = f'{runs_dir}/eval_metrics_ood.json'

if os.path.exists(eval_path):
    with open(eval_path) as f:
        em = json.load(f)
    print('=== 메인 평가 결과 ===')
    print(f'  Accuracy:          {em["accuracy"]:.4f}')
    print(f'  F1:                {em["f1"]:.4f}')
    print(f'  Perplexity:        {em["perplexity"]:.4f}')
    print(f'  ECE:               {em["ece"]:.4f}')
    print(f'  Sparsity:          {em["parameter_sparsity"]*100:.2f}%')
    print(f'  Speedup vs Teacher:{em["speedup_vs_teacher"]:.3f}x')
    print(f'  Compression:       {em["compression_ratio"]:.3f}x')
    print(f'  Latency:           {em["latency_ms"]:.1f} ms')

if os.path.exists(ood_path):
    with open(ood_path) as f:
        ood = json.load(f)
    print('\n=== OOD 평가 결과 (BioASQ) ===')
    print(f'  Accuracy:          {ood["accuracy"]:.4f}')
    print(f'  Samples:           {ood["samples_measured"]}')

---
## 7. Baseline 실험 (5종)

| Baseline | 설명 | 논문에서의 역할 |
|---|---|---|
| `zero_shot` | 학습 없음 (하한선) | Lower bound |
| `student_sft` | CE only, KD 없음 | KD 효과 검증 |
| `lora_only` | 도메인 적응만 | Domain adaptation 효과 |
| `direct_kd` | 적응 없이 바로 KD | Progressive 효과 |
| `no_distill_prune` | KD 없이 pruning | 蒸留+압축 결합 효과 |

In [ ]:
EXPERIMENT_SCALE = 'full'  # ★
config_path = f'/content/dapd/configs/dapd_colab_{EXPERIMENT_SCALE}.yaml'

print('🚀 Baseline 실험 시작 (5종)')
print('   예상 시간: ~4시간 (full scale A100 기준)')

!PYTHONPATH=/content/dapd/src \
  HF_HOME=/content/dapd/cache/hf \
  TOKENIZERS_PARALLELISM=false \
  python /content/dapd/scripts/run_baselines.py \
    --config {config_path} \
    --baselines zero_shot,student_sft,direct_kd,lora_only,no_distill_prune \
  2>&1 | tee /content/drive/MyDrive/Domain_Adaption/runs/baselines_log.txt

In [ ]:
# Baseline 결과 요약
import json, os

EXPERIMENT_SCALE = 'full'  # ★
baseline_root = f'/content/dapd/runs/dapd_{EXPERIMENT_SCALE}/baselines'

summary_path = os.path.join(baseline_root, 'baseline_summary.json')
if os.path.exists(summary_path):
    with open(summary_path) as f:
        bs = json.load(f)

    print(f'{"Baseline":<20} {"Accuracy":>10} {"F1":>8} {"PPL":>8} {"ECE":>8}')
    print('-' * 60)
    for name, result in bs.items():
        ev = result.get('evaluation') or {}
        acc = ev.get('accuracy', 'N/A')
        f1  = ev.get('f1', 'N/A')
        ppl = ev.get('perplexity', 'N/A')
        ece = ev.get('ece', 'N/A')
        print(f'{name:<20} {acc:>10.4f} {f1:>8.4f} {ppl:>8.4f} {ece:>8.4f}')

---
## 8. Ablation Study

| Ablation | 제거 요소 | 검증 내용 |
|---|---|---|
| `no_adapt` | 도메인 적응 단계 | 적응 전략의 효과 |
| `no_kd` | KL Divergence | 지식 증류의 효과 |
| `no_prune` | 구조적 가지치기 | 압축 효과 |
| `constant_temp` | Progressive Temperature | 온도 스케줄 효과 |

In [ ]:
EXPERIMENT_SCALE = 'full'  # ★
config_path = f'/content/dapd/configs/dapd_colab_{EXPERIMENT_SCALE}.yaml'

print('🚀 Ablation study 시작 (4종)')

!PYTHONPATH=/content/dapd/src \
  HF_HOME=/content/dapd/cache/hf \
  TOKENIZERS_PARALLELISM=false \
  python /content/dapd/scripts/run_ablation.py \
    --config {config_path} \
  2>&1 | tee /content/drive/MyDrive/Domain_Adaption/runs/ablation_log.txt

In [ ]:
# Ablation 결과 요약
import json, os

EXPERIMENT_SCALE = 'full'  # ★
ablation_root = f'/content/dapd/runs/dapd_{EXPERIMENT_SCALE}/ablation'

summary_path = os.path.join(ablation_root, 'ablation_summary.json')
if os.path.exists(summary_path):
    with open(summary_path) as f:
        ab = json.load(f)

    print(f'{"Variant":<22} {"Accuracy":>10} {"F1":>8} {"PPL":>8} {"Speedup":>9}')
    print('-' * 65)
    for name, result in ab.items():
        ev = result.get('evaluation') or {}
        acc = ev.get('accuracy', 'N/A')
        f1  = ev.get('f1', 'N/A')
        ppl = ev.get('perplexity', 'N/A')
        spd = ev.get('speedup_vs_teacher', 'N/A')
        print(f'{name:<22} {acc:>10.4f} {f1:>8.4f} {ppl:>8.4f} {spd:>9.3f}x')

---
## 9. Multi-Seed 실험 (논문 통계적 신뢰성)

3개 seed(42, 123, 777)로 반복 실행해 평균과 표준편차를 보고합니다.
이는 top conference 논문의 필수 요구사항입니다.

In [ ]:
EXPERIMENT_SCALE = 'full'  # ★
config_path = f'/content/dapd/configs/dapd_colab_{EXPERIMENT_SCALE}.yaml'

print('🚀 Multi-seed 실험 시작 (seed: 42, 123, 777)')
print('   예상 시간: A100 ~2.5시간')

!PYTHONPATH=/content/dapd/src \
  HF_HOME=/content/dapd/cache/hf \
  TOKENIZERS_PARALLELISM=false \
  python /content/dapd/scripts/run_multi_seed.py \
    --config {config_path} \
    --seeds 42,123,777 \
  2>&1 | tee /content/drive/MyDrive/Domain_Adaption/runs/multiseed_log.txt

In [ ]:
# Multi-seed 결과 요약 (평균 ± 표준편차)
import json, os, statistics

EXPERIMENT_SCALE = 'full'  # ★
multiseed_root = f'/content/dapd/runs/dapd_{EXPERIMENT_SCALE}/multi_seed'

accs, f1s, ppls = [], [], []
seeds = [42, 123, 777]

print(f'{"Seed":<8} {"Accuracy":>10} {"F1":>8} {"PPL":>8}')
print('-' * 40)

for seed in seeds:
    eval_path = os.path.join(multiseed_root, f'seed_{seed}', 'eval_metrics.json')
    if os.path.exists(eval_path):
        with open(eval_path) as f:
            ev = json.load(f)
        acc = ev.get('accuracy', 0)
        f1  = ev.get('f1', 0)
        ppl = ev.get('perplexity', 0)
        accs.append(acc); f1s.append(f1); ppls.append(ppl)
        print(f'{seed:<8} {acc:>10.4f} {f1:>8.4f} {ppl:>8.4f}')

if len(accs) >= 2:
    print('-' * 40)
    print(f'{"Mean":<8} {statistics.mean(accs):>10.4f} {statistics.mean(f1s):>8.4f} {statistics.mean(ppls):>8.4f}')
    print(f'{"Std":<8} {statistics.stdev(accs):>10.4f} {statistics.stdev(f1s):>8.4f} {statistics.stdev(ppls):>8.4f}')

---
## 10. 결과 Drive 백업 및 최종 정리

In [ ]:
# runs/가 Drive에 심볼릭 링크로 연결되어 있어 자동 저장됩니다.
# 이 셀은 최종 결과 파일 목록만 출력합니다.
import os

EXPERIMENT_SCALE = 'full'  # ★
runs_dir = f'/content/dapd/runs/dapd_{EXPERIMENT_SCALE}'

print('=== 저장된 결과 파일 목록 ===')
for root, dirs, files in os.walk(runs_dir):
    # 체크포인트 폴더 생략
    dirs[:] = [d for d in dirs if not d.startswith('checkpoint-')]
    level = root.replace(runs_dir, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = '  ' * (level + 1)
    for file in files:
        if file.endswith('.json'):
            fpath = os.path.join(root, file)
            size = os.path.getsize(fpath) / 1024
            print(f'{subindent}{file} ({size:.1f} KB)')

print('\n✅ 모든 결과가 Google Drive에 저장되었습니다.')
print(f'   Drive 경로: {DRIVE_PROJECT_PATH}/runs/')

---
## 트러블슈팅

### CUDA Out of Memory
```python
# 셀 맨 위에 추가
import torch
torch.cuda.empty_cache()
```

### T4에서 full-scale 실행 시 메모리 부족
- `EXPERIMENT_SCALE = 'v2'` 로 전환 (5k 샘플, 128 토큰)
- `per_device_train_batch_size: 2` 로 줄이기

### 세션 종료 후 재시작
```
1. 섹션 1 (Drive 마운트) 재실행
2. 섹션 2 (디렉토리 설정) 재실행  
3. 섹션 3 (패키지 설치) 재실행
4. 중단된 실험 셀만 재실행 (체크포인트에서 자동 재개)
```

### HuggingFace 다운로드 실패
```python
# HF 토큰 설정 (private 모델 접근 시)
from huggingface_hub import login
login(token='hf_YOUR_TOKEN')
```